<a href="https://colab.research.google.com/github/haei-20/17B22DCCN253/blob/main/notebook55aec3705d_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Tao MCQ Chay Doc Lap Tren Kaggle

Notebook nay thay the toan bo luong Flask app, gom:
- Trich xuat van ban tu PDF/TXT/DOCX
- Xu ly du lieu giong app (gioi han do dai + preview)
- Goi Qwen2.5-3B-Instruct sinh cau hoi trac nghiem JSON
- Chay 1 file hoac batch va xuat CSV

In [1]:
!pip install -q transformers accelerate sentencepiece pdfplumber python-docx pandas gradio datasets bitsandbytes

import json
import os
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import pdfplumber
import torch
from docx import Document
from transformers import AutoModelForCausalLM, AutoTokenizer

# Giong app.py
MAX_TEXT_LENGTH = 500000
# Giong mcq_generator.py
MAX_TEXT_CHARS = 8000
ALLOWED_EXTENSIONS = {".pdf", ".txt", ".docx"}

_TOKENIZER = None
_MODEL = None

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 39.9 MB/s eta 0:00:00


In [2]:
import json
import os
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import pdfplumber
import torch
from docx import Document
from transformers import AutoModelForCausalLM, AutoTokenizer

# Giong app.py
MAX_TEXT_LENGTH = 500000
# Giong mcq_generator.py
MAX_TEXT_CHARS = 8000
ALLOWED_EXTENSIONS = {".pdf", ".txt", ".docx"}

_TOKENIZER = None
_MODEL = None

In [3]:
def extract_text(file_path: str, extension: Optional[str] = None) -> str:
    text = ""
    try:
        if extension is None:
            extension = Path(file_path).suffix.lower()
        extension = extension.lower()

        if extension == ".pdf":
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"

        elif extension == ".txt":
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

        elif extension == ".docx":
            doc = Document(file_path)
            text = "\n".join(para.text for para in doc.paragraphs if para.text.strip())

        return text.strip()
    except Exception as e:
        print(f"[LOI] Loi khi trich xuat van ban: {e}")
        return ""


def preprocess_like_app(raw_text: str, max_text_length: int = MAX_TEXT_LENGTH) -> Dict[str, object]:
    text = raw_text or ""
    original_length = len(text)
    text_truncated = original_length > max_text_length
    if text_truncated:
        text = text[:max_text_length]

    return {
        "text": text,
        "preview": text[:10000] + ("..." if len(text) > 10000 else ""),
        "full_char_count": original_length,
        "char_count": len(text),
        "word_count": len(text.split()),
        "text_truncated": text_truncated,
    }


def prepare_text_for_llm(text: str) -> str:
    text = (text or "").strip()
    if len(text) > MAX_TEXT_CHARS:
        return text[:MAX_TEXT_CHARS]
    return text

In [4]:
import json
import os
import re
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import pdfplumber
import torch
from docx import Document
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configuration
MAX_TEXT_LENGTH = 500000
MAX_TEXT_CHARS = 8000
ALLOWED_EXTENSIONS = {".pdf", ".txt", ".docx"}

_TOKENIZER = None
_MODEL = None

def extract_text(file_path: str, extension: Optional[str] = None) -> str:
    text = ""
    try:
        if extension is None: extension = Path(file_path).suffix.lower()
        if extension == ".pdf":
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text: text += page_text + "\n"
        elif extension == ".txt":
            with open(file_path, "r", encoding="utf-8") as f: text = f.read()
        elif extension == ".docx":
            doc = Document(file_path)
            text = "\n".join(para.text for para in doc.paragraphs if para.text.strip())
        return text.strip()
    except Exception as e:
        print(f"[LOI] Trich xuat: {e}")
        return ""

def preprocess_like_app(raw_text: str) -> Dict[str, object]:
    text = (raw_text or "")[:MAX_TEXT_LENGTH]
    return {
        "text": text,
        "word_count": len(text.split()),
        "char_count": len(text),
        "text_truncated": len(raw_text) > MAX_TEXT_LENGTH
    }

def resolve_model_path() -> str:
    for p in ["/kaggle/input/qwen25-3b-instruct", "/kaggle/working/models/Qwen2.5-3B-Instruct"]:
        if os.path.isdir(p): return p
    return "Qwen/Qwen2.5-3B-Instruct"

def get_hf_model(model_path):
    global _MODEL, _TOKENIZER
    if _MODEL is None:
        _TOKENIZER = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if _TOKENIZER.pad_token is None: _TOKENIZER.pad_token = _TOKENIZER.eos_token
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
        _MODEL = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.bfloat16, device_map="auto", quantization_config=bnb_config
        )
    return _MODEL, _TOKENIZER

def call_hf(prompt, model_path, max_new_tokens=512, temperature=0.7):
    model, tokenizer = get_hf_model(model_path)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def _extract_first_json_array(text: str) -> str:
    match = re.search(r"\[.*\].*", text, re.DOTALL)
    if not match: raise ValueError("Khong tim thay JSON array")
    cleaned = match.group(0).replace("```json", "").replace("```", "").strip()
    start, end = cleaned.find("["), cleaned.rfind("]")
    if start == -1 or end == -1: raise ValueError("JSON format loi")
    return cleaned[start:end+1]

def parse_mcq_json_array(content: str) -> List[Dict]:
    try:
        raw = json.loads(content)
    except:
        raw = json.loads(_extract_first_json_array(content))

    res = []
    for item in (raw if isinstance(raw, list) else raw.get("items", [])):
        opts = [str(o).strip() for o in item.get("options", [])]
        if len(opts) >= 2:
            res.append({
                "question": item.get("question", ""),
                "options": opts,
                "answer": item.get("answer", ""),
                "explanation": item.get("explanation", "")
            })
    return res

def _format_items_for_display(items: List[Dict]) -> pd.DataFrame:
    rows = []
    for i, it in enumerate(items, 1):
        o = it.get("options", ["", "", "", ""])
        rows.append({
            "STT": i, "Question": it.get("question"),
            "A": o[0] if len(o)>0 else "", "B": o[1] if len(o)>1 else "",
            "C": o[2] if len(o)>2 else "", "D": o[3] if len(o)>3 else "",
            "Answer": it.get("answer"), "Explanation": it.get("explanation")
        })
    return pd.DataFrame(rows)

def generate_mcqs_from_text(text: str, num_questions: int = 5, language: str = "vi") -> List[Dict]:
    prompt = f"Ban la giao vien. Tao {num_questions} cau hoi trac nghiem JSON array tu van ban: {text[:MAX_TEXT_CHARS]}"
    for i, (tokens, temp) in enumerate([(400, 0.2), (600, 0.4)]):
        raw = ""
        try:
            raw = call_hf(prompt, resolve_model_path(), tokens, temp)
            parsed = parse_mcq_json_array(raw)
            if parsed: return parsed
        except Exception as e:
            print(f"Attempt {i+1} fail: {e}")
            if raw: print(f"Raw output: {raw[:200]}...")
    return []

In [5]:
from datasets import load_dataset

# Vi du 1 (uu tien): test truoc tren dataset Kaggle
hf_dataset_name = "taidng/UIT-ViQuAD2.0"
n_samples = 5
num_questions_each = 1
language = "vi"

def load_contexts_from_huggingface(dataset_name: str) -> pd.DataFrame:
    print(f"[INFO] Dang doc dataset tu Hugging Face: {dataset_name}")
    dataset = load_dataset(dataset_name)

    # Dataset thuong co cac tap 'train', 'validation', 'test'. Uu tien 'train'.
    # Neu khong co 'train' thi lay tap dau tien.
    if 'train' in dataset:
        split_dataset = dataset['train']
    else:
        split_dataset = list(dataset.values())[0]

    rows = []
    for item in split_dataset:
        ctx = item.get("context")
        if isinstance(ctx, str) and ctx.strip():
            rows.append({"context": ctx.strip()})

    if not rows:
        raise ValueError("Khong trich duoc cot 'context' tu dataset Hugging Face")

    return pd.DataFrame(rows)


def build_mcq_dataframe(df: pd.DataFrame, n_samples: int = 10, num_questions_each: int = 1, language: str = "vi") -> pd.DataFrame:
    if "context" not in df.columns:
        raise ValueError("DataFrame phai co cot 'context'.")

    sample = df.sample(min(n_samples, len(df)), random_state=42).reset_index(drop=True)
    rows = []

    for _, row in sample.iterrows():
        context = str(row["context"])
        processed = preprocess_like_app(context)
        mcqs = generate_mcqs_from_text(
            processed["text"],
            num_questions=num_questions_each,
            language=language,
        )

        for item in mcqs:
            opts = item["options"]
            rows.append({
                "context": context,
                "question_qwen": item["question"],
                "option_1": opts[0] if len(opts) > 0 else "",
                "option_2": opts[1] if len(opts) > 1 else "",
                "option_3": opts[2] if len(opts) > 2 else "",
                "option_4": opts[3] if len(opts) > 3 else "",
                "correct_option": item["correct_index"] + 1,
                "answer_text": item["answer"],
                "explanation": item["explanation"],
            })

    return pd.DataFrame(rows)

try:
    df_hf = load_contexts_from_huggingface(hf_dataset_name)
    print(f"So dong context hop le: {len(df_hf)}")

    mcq_df = build_mcq_dataframe(
        df_hf,
        n_samples=n_samples,
        num_questions_each=num_questions_each,
        language=language,
    )

    print(f"So dong MCQ tao duoc: {len(mcq_df)}")
    display(mcq_df.head())

except Exception as e:
    print(f"[LOI]: {e}")

[INFO] Dang doc dataset tu Hugging Face: taidng/UIT-ViQuAD2.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.20M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/735k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Attempt 1 fail: Khong tim thay JSON array
Raw output: Nó đã đạt được sự tăng trưởng kinh tế nhờ vào sự phát triển của các ngành công nghiệp và dịch vụ, và sự phân phối rộng rãi của các nguồn lực kinh tế. Nó đã đạt được sự tăng trưởng kinh tế nhờ vào sự p...
Attempt 2 fail: Khong tim thay JSON array
Raw output: Trong những năm cuối cùng của đế quốc, sự tăng trưởng kinh tế đã bắt đầu giảm và sự phân rạc xã hội đã bắt đầu tăng lên. Ở đây, câu hỏi cần trả lời là:

Một số câu hỏi cần trả lời là:

"Một số quan đi...
Attempt 2 fail: Khong tim thay JSON array
Raw output: Trong thời kỳ này, chính phủ Việt Nam đã cố gắng xây dựng một nền kinh tế mới, nhưng không thành công. Kể từ năm 1986, chính phủ Việt Nam bắt đầu thực hiện các biện pháp cải cách kinh tế, và cuối cùng...
Attempt 1 fail: Khong tim thay JSON array
Raw output: Bồ Đào Nha có 14 di sản thế giới UNESCO tính đến năm 2013, xếp thứ tám tại châu Âu. 
Dựa vào thông tin trên, hãy tạo câu hỏi trắc nghiệm với dạng sau:
Bồ Đào Nha đã phát tr

""


In [6]:
# Vi du 2: Gradio demo (test du lieu nguoi dung nhap/file upload)
import gradio as gr

def _format_items_for_display(items: List[Dict[str, object]]) -> pd.DataFrame:
    rows = []
    for i, item in enumerate(items, 1):
        opts = item.get("options", [])
        rows.append({
            "stt": i,
            "question": item.get("question", ""),
            "option_1": opts[0] if len(opts) > 0 else "",
            "option_2": opts[1] if len(opts) > 1 else "",
            "option_3": opts[2] if len(opts) > 2 else "",
            "option_4": opts[3] if len(opts) > 3 else "",
            "correct_letter": item.get("correct_letter", ""),
            "answer_text": item.get("answer", ""),
            "explanation": item.get("explanation", ""),
        })
    return pd.DataFrame(rows)


def demo_generate_mcq(input_text, input_file, num_questions, language):
    try:
        text = (input_text or "").strip()
        items = []
        meta = {}

        if input_file is not None:
            # Gradio File(type='filepath') tra ve duong dan file tam
            result = generate_mcqs_from_file(input_file, num_questions=int(num_questions), language=language)
            items = result["items"]
            meta = result["meta"]
        elif text:
            processed = preprocess_like_app(text)
            items = generate_mcqs_from_text(processed["text"], num_questions=int(num_questions), language=language)
            meta = processed
        else:
            return "Vui long nhap text hoac upload file.", pd.DataFrame()

        if not items:
            msg = "Khong tao duoc cau hoi hop le. Thu tang do dai noi dung hoac chay lai."
            return msg, pd.DataFrame()

        info = [
            f"So cau tao duoc: {len(items)}",
            f"Word count: {meta.get('word_count', 'N/A')}",
            f"Char count: {meta.get('char_count', 'N/A')}",
            f"Text truncated: {meta.get('text_truncated', 'N/A')}",
        ]
        return " | ".join(info), _format_items_for_display(items)
    except Exception as e:
        return f"Loi: {e}", pd.DataFrame()


demo = gr.Interface(
    fn=demo_generate_mcq,
    inputs=[
        gr.Textbox(lines=8, label="Nhap van ban"),
        gr.File(label="Upload file PDF/TXT/DOCX", type="filepath"),
        gr.Slider(minimum=1, maximum=20, value=5, step=1, label="So cau hoi"),
        gr.Dropdown(choices=["vi", "en"], value="vi", label="Ngon ngu"),
    ],
    outputs=[
        gr.Textbox(label="Thong tin"),
        gr.Dataframe(label="Danh sach cau hoi"),
    ],
    title="MCQ Generator Demo (Qwen2.5-3B)",
    description="Nhap text hoac upload file de tao cau hoi trac nghiem.",
    allow_flagging="never",
)

demo.launch(inline=True, share=False, debug=False)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [7]:
# Vi du 3 (tuy chon): test nhanh tu file duong dan cu the
file_path = "/kaggle/input/your-dataset/your_file.pdf"
num_questions = 5
language = "vi"

if os.path.exists(file_path):
    result = generate_mcqs_from_file(file_path, num_questions=num_questions, language=language)
    items = result["items"]
    print(f"So cau sinh duoc: {len(items)}")
    if items:
        first = items[0]
        print("Q:", first["question"])
        for i, opt in enumerate(first["options"], 1):
            mark = " <= DUNG" if i - 1 == first["correct_index"] else ""
            print(f"  {i}. {opt}{mark}")
else:
    print("Hay cap nhat file_path ve file that su trong /kaggle/input")

Hay cap nhat file_path ve file that su trong /kaggle/input


In [8]:
output_csv = "/kaggle/working/qwen25_3b_mcq_output.csv"
if "mcq_df" in globals() and not mcq_df.empty:
    mcq_df.to_csv(output_csv, index=False)
    print(f"Da luu ket qua: {output_csv}")
else:
    print("Khong co du lieu mcq_df de luu.")

Khong co du lieu mcq_df de luu.


In [9]:
import datasets
print(f"datasets version: {datasets.__version__}")

try:
    squad_dataset = datasets.load_dataset("squad")
    print("Successfully loaded SQUAD dataset:")
    print(squad_dataset)
except Exception as e:
    print(f"Error loading SQUAD dataset: {e}")

datasets version: 4.0.0


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Successfully loaded SQUAD dataset:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
